<a href="https://colab.research.google.com/github/Md-Istiaq/Pre_Trained_Baseline_model-XLM_RoBERTa-_implementation/blob/main/Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary packages
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install matplotlib seaborn

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import XLMRobertaTokenizer, TFXLMRobertaForSequenceClassification, pipeline
import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# Load dataset
df = pd.read_excel('Balanced_BanglaSpamData.xlsx')

# Display first few rows of the dataset
df.head()


,v1,v2
0,ham,আজকে brainstorming session arrange করেছি।
1,spam,আপনার জন্য দারুণ সুযোগ! কুইজে অংশগ্রহণ করুন এব...
2,ham,"I’ll get back to you after my meeting, এখন ব্য..."
3,spam,Fantasy Football খেলুন এবং Free Xiaomi Mi Band...
4,ham,"""Don't forget our meeting tomorrow. কালকের মিট..."


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import IPython.display as display

# Load the dataset
file_path = 'Balanced_BanglaSpamData.xlsx'  # Update with the correct file path
df = pd.read_excel(file_path)
df.rename(columns={'v1': 'Label', 'v2': 'Text'}, inplace=True)
# Prepare the sample data
df_sample = df.head(10)  # Display first 10 rows for the table
#df_sample['v1'] = df_sample['v2']  # Assuming 'cleaned_text' column exists or you preprocess it

# Display the DataFrame (optional for Jupyter Notebook)
display.display(df_sample)

# Create a Plotly table
fig = go.Figure(data=[go.Table(
    header=dict(values=list(df_sample.columns),
                fill_color='midnightblue',
                font=dict(color='white'),
                align='left'),
    cells=dict(values=[df_sample.Label, df_sample.Text],  # Adjust as per columns
               fill_color='snow',
               align='left'))
])

# Update layout for better presentation
fig.update_layout(
    title='<span style="font-size:24px; font-family:Times New Roman"> First few row of Dataset</span>',
    width=800,  # Width of the table
    height=680  # Height of the table
)

# Show the plot
fig.show()


,Label,Text
0,ham,আজকে brainstorming session arrange করেছি।
1,spam,আপনার জন্য দারুণ সুযোগ! কুইজে অংশগ্রহণ করুন এব...
2,ham,"I’ll get back to you after my meeting, এখন ব্য..."
3,spam,Fantasy Football খেলুন এবং Free Xiaomi Mi Band...
4,ham,"""Don't forget our meeting tomorrow. কালকের মিট..."
5,spam,ফ্রি পিক্সেলআর্ট সাবস্ক্রিপশন পেতে visit করুন:...
6,ham,"I'll email you the details later, আজ একটু ব্যস্ত।"
7,spam,আপনার বিকাশ একাউন্টে ফ্রি ১০০০ টাকা যোগ করতে v...
8,ham,আজকে dinner plan করেছি family এর সাথে।
9,spam,Top Cricket Champs এ যোগ দিন এবং পুরস্কার জিতু...


In [ ]:

import nltk
nltk.download('stopwords')

import re
from nltk.corpus import stopwords

# ... (rest of your code remains the same) ...
# Define Bangla stopwords
bangla_stopwords = [
    "অবশ্য", "অনেক", "অনেকে", "অনেকেই", "অন্তত", "অথবা", "অথচ", "অর্থাত", "অন্য", "আজ", "আছে",
    "আপনার", "আপনি", "আবার", "আমরা", "আমাকে", "আমাদের", "আমার", "আমি", "আরও", "আর", "আগে", "আগেই",
    "আই", "অতএব", "আগামী", "অবধি", "অনুযায়ী", "আদ্যভাগে", "এই", "একই", "একে", "একটি", "এখন", "এখনও",
    "এখানে", "এখানেই", "এটি", "এটা", "এটাই", "এতটাই", "এবং", "একবার", "এবার", "এদের", "এঁদের",
    "এমন", "এমনকী", "এল", "এর", "এরা", "এঁরা", "এস", "এত", "এতে", "এসে", "একে", "এ", "ঐ", "ই",
    "ইহা", "ইত্যাদি", "উনি", "উপর", "উপরে", "উচিত", "ও", "ওই", "ওর", "ওরা", "ওঁর", "ওঁরা", "ওকে",
    "ওদের", "ওঁদের", "ওখানে", "কত", "কবে", "করতে", "কয়েক", "কয়েকটি", "করবে", "করলেন", "করার", "কারও",
    "করা", "করি", "করিয়ে", "করার", "করাই", "করলে", "করলেন", "করিতে", "করিয়া", "করেছিলেন", "করছে",
    "করছেন", "করেছেন", "করেছে", "করেন", "করবেন", "করায়", "করে", "করেই", "কাছ", "কাছে", "কাজে", "কারণ",
    "কিছু", "কিছুই", "কিন্তু", "কিংবা", "কি", "কী", "কেউ", "কেউই", "কাউকে", "কেন", "কে", "কোনও", "কোনো",
    "কোন", "কখনও", "ক্ষেত্রে", "খুব", "গুলি", "গিয়ে", "গিয়েছে", "গেছে", "গেল", "গেলে", "গোটা", "চলে",
    "ছাড়া", "ছাড়াও", "ছিলেন", "ছিল", "জন্য", "জানা", "ঠিক", "তিনি", "তিনঐ", "তিনিও", "তখন", "তবে", "তবু",
    "তাঁদের", "তাঁাহারা", "তাঁরা", "তাঁর", "তাঁকে", "তাই", "তেমন", "তাকে", "তাহা", "তাহাতে", "তাহার",
    "তাদের", "তারপর", "তারা", "তারৈ", "তার", "তাহলে", "তিনি", "তা", "তাও", "তাতে", "তো", "তত", "তুমি",
    "তোমার", "তথা", "থাকে", "থাকা", "থাকায়", "থেকে", "থেকেও", "থাকবে", "থাকেন", "থাকবেন", "থেকেই", "দিকে",
    "দিতে", "দিয়ে", "দিয়েছে", "দিয়েছেন", "দিলেন", "দু", "দুটি", "দুটো", "দেয়", "দেওয়া", "দেওয়ার", "দেখা",
    "দেখে", "দেখতে", "দ্বারা", "ধরে", "ধরা", "নয়", "নানা", "না", "নাকি", "নাগাদ", "নিতে", "নিজে", "নিজেই",
    "নিজের", "নিজেদের", "নিয়ে", "নেওয়া", "নেওয়ার", "নেই", "নাই", "পক্ষে", "পর্যন্ত", "পাওয়া", "পারেন",
    "পারি", "পারে", "পরে", "পরেই", "পরেও", "পর", "পেয়ে", "প্রতি", "প্রভৃতি", "প্রায়", "ফের", "ফলে",
    "ফিরে", "ব্যবহার", "বলতে", "বললেন", "বলেছেন", "বলল", "বলা", "বলেন", "বলে", "বহু", "বসে", "বার",
    "বা", "বিনা", "বরং", "বদলে", "বাদে", "বার", "বিশেষ", "বিভিন্ন", "বিষয়টি", "ব্যবহার", "ব্যাপারে", "ভাবে",
    "ভাবেই", "মধ্যে", "মধ্যেই", "মধ্যেও", "মধ্যভাগে", "মাধ্যমে", "মাত্র", "মতো", "মতোই", "মোটেই", "যখন",
    "যদি", "যদিও", "যাবে", "যায়", "যাকে", "যাওয়া", "যাওয়ার", "যত", "যতটা", "যা", "যার", "যারা",
    "যাঁর", "যাঁরা", "যাদের", "যান", "যাচ্ছে", "যেতে", "যাতে", "যেন", "যেমন", "যেখানে", "যিনি", "যে",
    "রেখে", "রাখা", "রয়েছে", "রকম", "শুধু", "সঙ্গে", "সঙ্গেও", "সমস্ত", "সব", "সবার", "সহ", "সুতরাং",
    "সহিত", "সেই", "সেটা", "সেটি", "সেটাই", "সেটাও", "সম্প্রতি", "সেখান", "সেখানে", "সে", "স্পষ্ট", "স্বয়ং",
    "হইতে", "হইবে", "হৈলে", "হইয়া", "হচ্ছে", "হত", "হতে", "হতেই", "হবে", "হবেন", "হয়েছিল", "হয়েছে",
    "হয়েছেন", "হয়ে", "হয়নি", "হয়", "হয়েই", "হয়তো", "হল", "হলে", "হলেই", "হলেও", "হলো", "হিসাবে", "হওয়া",
    "হওয়ার", "হওয়ায়", "হন", "হোক", "জন", "জনকে", "জনের", "জানতে", "জানায়", "জানিয়ে", "জানানো", "জানিয়েছে",
    "জন্য", "জন্যওজে", "জে", "বেশ", "দেন", "তুলে", "ছিলেন", "চান", "চায়", "চেয়ে", "মোট", "যথেষ্ট", "টি"
]

def clean_text(text):
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Convert to lowercase for English
    text = text.lower()
    # Combine Bangla and English stopwords
    stop_words = set(stopwords.words('english')) | set(bangla_stopwords)
    # Remove stopwords
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Apply text cleaning
df['cleaned_text'] = df['v2'].apply(clean_text)

# Display the dataset after cleaning
print(df.head())


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


     v1                                                 v2  \
0   ham          আজকে brainstorming session arrange করেছি।   
1  spam  আপনার জন্য দারুণ সুযোগ! কুইজে অংশগ্রহণ করুন এব...   
2   ham  I’ll get back to you after my meeting, এখন ব্য...   
3  spam  Fantasy Football খেলুন এবং Free Xiaomi Mi Band...   
4   ham  "Don't forget our meeting tomorrow. কালকের মিট...   

                                        cleaned_text  
0              আজক brainstorming session arrange করছ  
1  আপনর জনয দরণ সযগ কইজ অশগরহণ করন এব জতন টক কযশব...  
2                      ill get back meeting বযসত রয়ছ  
3  fantasy football খলন এব free xiaomi mi band জত...  
4       dont forget meeting tomorrow কলকর মট ভল যও ন  


In [ ]:
# Split the dataset into train and test
X = df['cleaned_text']
y = df['v1'].map({'ham': 0, 'spam': 1})  # Assuming 'ham' = 0, 'spam' = 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check the shape of the split data
print(f'Training data size: {len(X_train)}')
print(f'Test data size: {len(X_test)}')


Training data size: 1658
Test data size: 415


In [ ]:
# Load the pre-trained XLM-RoBERTa tokenizer and model
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
model = TFXLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFXLMRobertaForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFXLMRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
import numpy as np

# Load the pre-trained XLM-RoBERTa tokenizer and model
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2)

# Tokenize the input data
def tokenize_function(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256)

train_encodings = tokenize_function(X_train.tolist())
test_encodings = tokenize_function(X_test.tolist())

# Prepare the dataset for PyTorch
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, y_train)
test_dataset = SpamDataset(test_encodings, y_test)

# Define the compute_metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    cm = confusion_matrix(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm.tolist(),  # Convert to list for easier display
    }

# Experiment with hyperparameters for tuning
hyperparameters = {
    "learning_rate": [5e-5, 3e-5],  # Different learning rates to try
    "num_train_epochs": [3, 5],     # Number of epochs
    "batch_size": [8, 16],          # Batch sizes
}

# Loop through hyperparameters
results = []
for lr in hyperparameters["learning_rate"]:
    for epochs in hyperparameters["num_train_epochs"]:
        for batch_size in hyperparameters["batch_size"]:
            print(f"Training with lr={lr}, epochs={epochs}, batch_size={batch_size}")

            # Initialize the TrainingArguments
            training_args = TrainingArguments(
                output_dir=f'./results_lr{lr}_ep{epochs}_bs{batch_size}',  # Output directory
                num_train_epochs=epochs,                                 # Number of training epochs
                per_device_train_batch_size=batch_size,                  # Training batch size
                per_device_eval_batch_size=batch_size,                   # Evaluation batch size
                learning_rate=lr,                                        # Learning rate
                warmup_steps=500,                                        # Warmup steps
                weight_decay=0.01,                                       # Weight decay
                logging_dir='./logs',                                    # Log directory
                logging_steps=10,                                        # Log every 10 steps
                evaluation_strategy="epoch",                             # Evaluate at the end of each epoch
                save_strategy="epoch",                                   # Save model at the end of each epoch
                save_total_limit=2,                                      # Limit number of saved models
                load_best_model_at_end=True                              # Load best model at the end of training
            )

            # Initialize the Trainer
            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=test_dataset,
                compute_metrics=compute_metrics,  # Attach metrics computation function
            )

            # Train the model
            trainer.train()

            # Evaluate the model
            metrics = trainer.evaluate()
            print(f"Results for lr={lr}, epochs={epochs}, batch_size={batch_size}:")
            print(metrics)

            # Store results
            results.append({
                "learning_rate": lr,
                "epochs": epochs,
                "batch_size": batch_size,
                "metrics": metrics,
            })

# Display all results
for result in results:
    print("\n------------------------------------------")
    print(f"Learning Rate: {result['learning_rate']}")
    print(f"Epochs: {result['epochs']}")
    print(f"Batch Size: {result['batch_size']}")
    print("Metrics:")
    for key, value in result["metrics"].items():
        if key == "confusion_matrix":
            print(f"{key}:\n{np.array(value)}")
        else:
            print(f"{key}: {value}")


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with lr=5e-05, epochs=3, batch_size=8


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning:

`evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

...

<ipython-input-26-d266d25a2509>:24: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Confusion Matrix
1,0.015000,0.129064,0.987952,0.990148,0.985294,0.987715,"[[209, 2], [3, 201]]"
2,0.000000,0.120869,0.987952,0.985366,0.990196,0.987775,"[[208, 3], [2, 202]]"
3,0.056100,0.130035,0.987952,0.990148,0.985294,0.987715,"[[209, 2], [3, 201]]"
4,0.000000,0.182736,0.983133,0.975845,0.990196,0.982968,"[[206, 5], [2, 202]]"
5,0.000000,0.119014,0.987952,0.990148,0.985294,0.987715,"[[209, 2], [3, 201]]"


Trainer is attempting to log a value of "[[209, 2], [3, 201]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
<ipython-input-26-d266d25a2509>:24: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Trainer is attempting to log a value of "[[208, 3], [2, 202]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
<ipython-input-26-d266d25a2509>:24: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Trainer is attempting to log a value of "[[209, 2], [3, 201]]" of type <class 

Trainer is attempting to log a value of "[[209, 2], [3, 201]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.


Results for lr=3e-05, epochs=5, batch_size=16:
{'eval_loss': 0.11901438236236572, 'eval_accuracy': 0.9879518072289156, 'eval_precision': 0.9901477832512315, 'eval_recall': 0.9852941176470589, 'eval_f1': 0.9877149877149877, 'eval_confusion_matrix': [[209, 2], [3, 201]], 'eval_runtime': 1.453, 'eval_samples_per_second': 285.621, 'eval_steps_per_second': 17.894, 'epoch': 5.0}

------------------------------------------
Learning Rate: 5e-05
Epochs: 3
Batch Size: 8
Metrics:
eval_loss: 0.08393179625272751
eval_accuracy: 0.9879518072289156
eval_precision: 0.9901477832512315
eval_recall: 0.9852941176470589
eval_f1: 0.9877149877149877
eval_confusion_matrix: [[209, 2], [3, 201]]
eval_runtime: 1.5973
eval_samples_per_second: 259.809
eval_steps_per_second: 32.554
epoch: 3.0

------------------------------------------
Learning Rate: 5e-05
Epochs: 3
Batch Size: 16
Metrics:
eval_loss: 0.08148369938135147
eval_accuracy: 0.9903614457831326
eval_precision: 0.9854368932038835
eval_recall: 0.9950980392156